# Your Personal Chatbot, Tetris 🤖
In this research page, students will be able to ask questions with my Python-written and Ollama-installed chatbot and receive answers so they can take notes on what they can expect to know and understand if interested in game design, game development, and software engineering.

In [5]:
import json
import glob
import os
import pandas as pd
import numpy as np
import ollama

# ==============================================================================
# 1. DYNAMICALLY EXTRACT ALL NOTEBOOKS & CSV DATA (INCLUDING TEXT COLUMNS)
# ==============================================================================
print("⚡ 1. Extracting data chunks from notebooks and datasets...")

def load_research_chunks():
    chunks = []
    
    # 1. Scan ALL .ipynb files (including subfolders like datasets/)
    notebook_files = glob.glob("**/*.ipynb", recursive=True)
    for nb_file in sorted(notebook_files):
        if "MyChatbot" in nb_file:
            continue
        try:
            with open(nb_file, 'r', encoding='utf-8') as f:
                nb_data = json.load(f)
            for cell in nb_data.get('cells', []):
                if cell.get('cell_type') == 'markdown':
                    text = "".join(cell.get('source', [])).strip()
                    if text:
                        chunks.append(f"[{nb_file} Notes]: {text}")
                elif cell.get('cell_type') == 'code':
                    code_text = "".join(cell.get('source', [])).strip()
                    if code_text:
                        chunks.append(f"[{nb_file} Code]: {code_text[:300]}")
        except Exception:
            continue

    # 2. Extract actual CSV rows (Gender, Future Career, Age, Interested Domain)
    csv_files = glob.glob("**/*.csv", recursive=True)
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            
            # Group demographics & careers into text summaries so NLP embeddings can search them
            if 'Gender' in df.columns and 'Future Career' in df.columns:
                career_counts = df.groupby(['Gender', 'Future Career']).size().reset_index(name='Count')
                summary_text = f"[{csv_file} Career Data by Gender]:\n" + career_counts.to_string(index=False)
                chunks.append(summary_text)
                
            # Add general CSV row sample
            chunks.append(f"[{csv_file} Dataset Preview]: Columns = {list(df.columns)}\nSample Rows:\n" + df.head(10).to_string())
        except Exception:
            pass
                
    return chunks

research_chunks = load_research_chunks()
print(f"✅ Extracted {len(research_chunks)} total research chunks.")

# ==============================================================================
# 2. GENERATE VECTOR EMBEDDINGS (NLP)
# ==============================================================================
print("🧠 2. Encoding all chunks into vector embeddings...")

chunk_vectors = []
for chunk in research_chunks:
    res = ollama.embeddings(model='nomic-embed-text', prompt=chunk)
    chunk_vectors.append(res['embedding'])

chunk_vectors = np.array(chunk_vectors)

# ==============================================================================
# 3. COSINE SIMILARITY SEARCH ALGORITHM (RAG)
# ==============================================================================
def cosine_similarity(vec_a, vec_b):
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

def retrieve_top_chunks(user_question, top_k=4):
    q_res = ollama.embeddings(model='nomic-embed-text', prompt=user_question)
    q_vec = np.array(q_res['embedding'])
    
    scores = [cosine_similarity(q_vec, c_vec) for c_vec in chunk_vectors]
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    return [(research_chunks[idx], scores[idx]) for idx in top_indices]

# ==============================================================================
# 4. QUERY CHATBOT
# ==============================================================================
user_question = input("\n💬 Ask Tetris a research question: ").strip()

if user_question:
    print("\n🔎 Searching notebook chunks and CSV career data...")
    top_results = retrieve_top_chunks(user_question, top_k=4)
    
    context_str = "\n\n".join([f"(Similarity: {score:.2f})\n{chunk}" for chunk, score in top_results])
    
    print(f"\n✅ Retrieved Top Context:\n{context_str}\n" + "="*60)
    print("🤖 Tetris is generating answer...")
    
    augmented_prompt = f"""
    Context retrieved from project CSV datasets and notebooks:
    
    {context_str}
    
    User Question: {user_question}
    
    Instruction: Answer using the exact dataset values above (mention specific roles like Software Engineer, Machine Learning Researcher, Information Security Analyst if listed in the data).
    """
    
    response = ollama.chat(
        model='llama3.2',
        messages=[
            {'role': 'system', 'content': 'You are Tetris, a RAG research assistant.'},
            {'role': 'user', 'content': augmented_prompt}
        ]
    )
    
    print("\n🤖 Tetris Answer:\n")
    print(response['message']['content'])

⚡ 1. Extracting data chunks from notebooks and datasets...
✅ Extracted 298 total research chunks.
🧠 2. Encoding all chunks into vector embeddings...



💬 Ask Tetris a research question:  What HBCUs are good for game development?



🔎 Searching notebook chunks and CSV career data...

✅ Retrieved Top Context:
(Similarity: 0.63)
[Gaming.ipynb Notes]: In conclusion,

here are the best coding languages to learn by platform.

1. PC & Console (AAA / High-Performance 3D)
Best Languages to Learn: C++ (Primary) & Lua (Secondary)

Target Genres: 3D Action-Adventure, Open-World RPGs, First-Person Shooters, Graphics-Heavy Simulation.

2. Mobile, VR & Cross-Platform Indie Games
Best Language to Learn: C#

Target Genres: 2D/3D Mobile Games, VR Experiences, Casual, Platformers, Indie RPGs.

Why: C# is the default language for Unity, making it fast to prototype, highly readable, and easily deployable across iOS, Android, PC, and consoles without managing complex manual memory management.

3. Native Android & Mobile-First Applications
Best Language to Learn: Java

Target Genres: 2D Arcade, Puzzle, Mobile Sandbox games (e.g., Minecraft Android roots).

4. Browser-Based & Web Applications
Best Language to Learn: JavaScript (HTML5 /

In [ ]:
import ollama

messages = []

print("🤖 Interactive Local AI Chat! (Type 'exit' or 'q' to stop)\n" + "─"*50)

while True:
    # Text box pops up for input
    user_input = input("💬 You: ").strip()
    
    if user_input.lower() in ['exit', 'quit', 'q']:
        print("Chat stopped.")
        break
        
    if not user_input:
        continue

    # Remember conversation history
    messages.append({'role': 'user', 'content': user_input})
    
    print("\n🤖 Thinking...")
    response = ollama.chat(model='llama3.2', messages=messages)
    reply = response['message']['content']
    
    # Store bot response in history
    messages.append({'role': 'assistant', 'content': reply})
    
    print(f"\n🤖 Answer:\n{reply}\n" + "─"*50 + "\n")

🤖 Interactive Local AI Chat! (Type 'exit' or 'q' to stop)
──────────────────────────────────────────────────


💬 You:  What public universities have good game development programs?



🤖 Thinking...

🤖 Answer:
Here are some public universities in the US with strong game development programs:

1. University of Southern California (USC) - Los Angeles, CA
 * USC's Interactive Media & Games Division is one of the top-ranked game development programs in the country.
2. Carnegie Mellon University - Pittsburgh, PA
 * Carnegie Mellon's School of Computer Science has a highly respected game development program that focuses on both artistic and technical aspects of game design.
3. Texas A&M University - College Station, TX
 * Texas A&M's Department of Computer Science offers a game development program with a focus on game engines, artificial intelligence, and human-computer interaction.
4. University of Washington - Seattle, WA
 * The University of Washington's Paul G. Allen School of Computer Science & Engineering has a strong game development program that emphasizes both technical and artistic skills.
5. University of Central Florida (UCF) - Orlando, FL
 * UCF's College of 